# Load and fatigue EDA (analytics/load)

Descriptive tables over the player-stint and player-game tables of `analytics.load.features` (docs/contract/load.md). Output is tables and markdown only. Held-out rows are excluded from every table.

Run from the repo root. With `--fixtures` style inputs (the default below) every number is **fixture output, not a result**; point `DATA_ROOT` at the owner's Mac layout for the real tables.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'analytics').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'tests'))

from analytics.load import eda, features as F, validate as V

DATA_ROOT = None   # e.g. '/Volumes/Drive/nba_harvest' on the Mac; None uses the synthetic season
inputs = V.load_data_root(DATA_ROOT) if DATA_ROOT else V.load_fixtures(seed=0, n_games=8, heldout_every=4, prefer_shared=False)
tables = F.build_tables(inputs, n_boot=200, seed=0)
stints, games, summary = tables['player_stints'], tables['player_games'], tables['load_summary']
print(inputs['source'], '| stints', len(stints), '| player-games', len(games), '| players', len(summary))
print(V.injuries_note(inputs.get('injuries')))

## Speed by stint index

Average speed of a stint against its index within the game (0 = the player's first stint). A within-game decline shows as a falling mean; short late stints inflate the noise.

In [ ]:
eda.speed_by_stint_index(stints)

## Decline by rest bucket

`decline_ftps` = average speed of the last qualifying stint minus the first. Buckets: back_to_back (1 day or less), one_day (2), two_plus (3 or more, including the first listed game of a team).

In [ ]:
eda.decline_by_rest_bucket(games)

## Distance distributions

Per player-game distance (ft), minutes, average speed and ft per minute; per stint distance and minutes.

In [ ]:
eda.distance_distribution(games, stints)

## Correlation table

Spearman correlations among the schedule and history features, the same-game totals and the targets. Same-game totals are never features; they are here to see what the target co-moves with.

In [ ]:
eda.correlation_table(games)

## Shuffled check

Rank correlation of each feature with `decline_ftps` on the real rows against the band of correlations when the target is permuted within game. A real value inside the band carries no more than chance.

In [ ]:
eda.shuffled_check(games, n_shuffles=100, seed=0)

## The stint gate

Players with fewer than `MIN_STINTS_RANK` stints are flagged insufficient and never ranked.

In [ ]:
print(eda.markdown(eda.stint_gate(summary)))
summary[~summary['insufficient']].sort_values('mean_decline').head(15)